In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("/content/01 covid_toy (1).csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
print(df.cough.value_counts())

print()
print()

print(df.city.value_counts())

cough
Mild      62
Strong    38
Name: count, dtype: int64


city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64


In [6]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [7]:
x = df.iloc[:,0:5]
y = df.drop(x,axis='columns')

print(x.head())

print()
print()
print()

print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [8]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)
print(x_train.head())

print()
print()
print()

print(y_train.head())

    age  gender  fever   cough       city
4    65  Female  101.0    Mild     Mumbai
99   10  Female   98.0  Strong    Kolkata
63   10    Male  100.0    Mild  Bangalore
95   12  Female  104.0    Mild  Bangalore
60   24  Female  102.0  Strong  Bangalore



   has_covid
4         No
99       Yes
63        No
95        No
60       Yes


In [9]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [10]:
si= SimpleImputer()
le= LabelEncoder()

In [11]:
y_train.has_covid = le.fit_transform(y_train.has_covid)
y_test.has_covid = le.transform(y_test.has_covid)

In [12]:
x_train['fever'] = si.fit_transform(x_train[['fever']])

In [13]:
from sklearn.preprocessing import OrdinalEncoder

In [14]:
oe = OrdinalEncoder()

x_train['cough'] = oe.fit_transform(x_train[['cough']])
x_test['cough'] = oe.transform(x_test[['cough']])

In [15]:
from sklearn.preprocessing import OneHotEncoder

In [16]:
ohe = OneHotEncoder(drop='first',sparse_output =False)

In [17]:
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city = ohe.transform(x_test[['gender','city']])

print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [18]:
x_train

,age,gender,fever,cough,city
4,65,Female,101.000000,0.0,Mumbai
99,10,Female,98.000000,1.0,Kolkata
63,10,Male,100.000000,0.0,Bangalore
95,12,Female,104.000000,0.0,Bangalore
60,24,Female,102.000000,1.0,Bangalore
...,...,...,...,...,...
43,22,Female,99.000000,0.0,Bangalore
19,42,Female,100.931507,1.0,Bangalore
14,51,Male,104.000000,0.0,Bangalore
53,83,Male,98.000000,0.0,Delhi


In [19]:
from sklearn.compose import ColumnTransformer

In [33]:
transformer = [
    ('trans1', SimpleImputer(strategy='most_frequent'), ['fever']),

    ('trans2', OneHotEncoder(
        drop='first',
        sparse_output=False,
        handle_unknown='ignore'
    ), ['gender', 'city'])
]

In [34]:
ct = ColumnTransformer(transformers=transformer,remainder='passthrough').set_output(transform='pandas')

In [35]:
x_train_trans = ct.fit_transform(x_train)
x_test_trans = ct.transform(x_test)

In [36]:
x_train_trans.sample(5)

print()
print()

print(x_train_trans.shape)



(80, 7)


In [37]:
x_test_trans = ct.transform(x_test)

In [38]:
x_test_trans.sample(5)

print()
print()

print(x_test_trans.shape)



(20, 7)


In [39]:
print(x_train_trans.head())
print(x_test_trans.head())

    trans1__fever  trans2__gender_Male  trans2__city_Delhi  \
4           101.0                  0.0                 0.0   
99           98.0                  0.0                 0.0   
63          100.0                  1.0                 0.0   
95          104.0                  0.0                 0.0   
60          102.0                  0.0                 0.0   

    trans2__city_Kolkata  trans2__city_Mumbai  remainder__age  \
4                    0.0                  1.0              65   
99                   1.0                  0.0              10   
63                   0.0                  0.0              10   
95                   0.0                  0.0              12   
60                   0.0                  0.0              24   

    remainder__cough  
4                0.0  
99               1.0  
63               0.0  
95               0.0  
60               1.0  
    trans1__fever  trans2__gender_Male  trans2__city_Delhi  \
82           98.0                  1